# Study 0 — Verification & Drive Backup

Seed-42 consistency check and result archiving.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

In [ ]:
import pickle

if ARTIFACT_PATH.exists():
    with open(ARTIFACT_PATH, 'rb') as f:
        artifact = pickle.load(f)
    print(f'Loaded preprocessing artifact from {ARTIFACT_PATH}')
    print(f'  Keys: {list(artifact.keys())}')
else:
    print(f'No preprocessing artifact at {ARTIFACT_PATH} — will build from scratch')
    artifact = None

## Seed-42 Consistency Check

In [ ]:
# ATS-217 verification: seed-42 scratch numbers must match across all result files
import json
from pathlib import Path

bench_path = RESULTS_DIR / 'final_benchmark.json'
multi_path = RESULTS_DIR / 'multiseed_results.json'
ssl_path   = RESULTS_DIR / 'ssl_experiment_results.json'

with open(bench_path) as f:
    bench = json.load(f)
with open(multi_path) as f:
    multi = json.load(f)
with open(ssl_path) as f:
    ssl = json.load(f)

# ── Config fingerprint check (catches stale caches) ──────────────────
fp_bench = bench.get('_meta', {}).get('config_fingerprint')
fp_multi = multi.get('_meta', {}).get('config_fingerprint')
fp_ssl   = ssl.get('_meta', {}).get('config_fingerprint')

print('Config fingerprint check')
print(f'  final_benchmark : {fp_bench}')
print(f'  multiseed       : {fp_multi}')
print(f'  ssl_experiment  : {fp_ssl}')

if fp_bench and fp_multi and fp_ssl:
    if fp_bench == fp_multi == fp_ssl:
        print('  -> All fingerprints match.\n')
    else:
        print('  -> MISMATCH! Result files were generated with different HP configs.')
        print('     Re-run ALL cells with FORCE_RERUN=True to regenerate from same sweep winner.\n')
        raise AssertionError('Config fingerprint mismatch -- stale cached results detected.')
else:
    print('  -> Some files lack fingerprints (pre-patch results). Skipping check.\n')

# ── AUROC consistency check ──────────────────────────────────────────
OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print('Seed-42 AUROC consistency check')
print('=' * 75)
all_ok = True
for oc in OUTCOMES:
    # final_benchmark: top-level model key -> outcome -> metric
    bench_auroc = bench.get('ft_scratch', {}).get(oc, {}).get('auroc')
    # multiseed: per-seed AUROC for seed 42
    multi_auroc = (multi.get('multiseed', {})
                        .get(oc, {})
                        .get('per_seed_auroc', {})
                        .get('42'))
    # ssl experiment: baseline AUROC
    ssl_auroc = ssl.get('baseline', {}).get(oc, {}).get('auroc')

    # Allow None==None for skipped outcomes (e.g. audit_qualification)
    match_bm = (bench_auroc is None and multi_auroc is None) or \
               (bench_auroc is not None and multi_auroc is not None and abs(bench_auroc - multi_auroc) < 1e-6)
    match_bs = (bench_auroc is None and ssl_auroc is None) or \
               (bench_auroc is not None and ssl_auroc is not None and abs(bench_auroc - ssl_auroc) < 1e-6)
    ok = match_bm and match_bs
    if not ok:
        all_ok = False
    status = 'PASS' if ok else 'FAIL'
    print(f'  {oc:25s}  bench={bench_auroc!s:>22s}  multi={multi_auroc!s:>22s}  ssl={ssl_auroc!s:>22s}  [{status}]')

print('=' * 75)
if all_ok:
    print('ALL PASS -- seed-42 scratch numbers are consistent across all files.')
    print('ATS-217 / C1 resolved. Clear for preprint.')
else:
    print('FAIL -- seed-42 numbers diverge. Re-run with FORCE_RERUN=True.')

assert all_ok, 'Seed-42 consistency check failed!'


## Save Results to Drive

In [ ]:
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/Fin-JEPA/results/study0'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Copy all result JSONs
for fname in ['benchmark_results.json', 'ft_sweep_results.json',
              'ssl_experiment_results_v2.json', 'final_benchmark.json',
              'multiseed_results.json', 'walk_forward_results.json']:
    src = RESULTS_DIR / fname
    if src.exists():
        shutil.copy2(str(src), DRIVE_RESULTS)
        print(f'Copied {fname} to Drive')

# Copy figures
fig_dir = RESULTS_DIR / 'figures'
if fig_dir.exists():
    drive_fig_dir = f'{DRIVE_RESULTS}/figures'
    shutil.copytree(str(fig_dir), drive_fig_dir, dirs_exist_ok=True)
    n_figs = len(list(fig_dir.glob('*.png')))
    print(f'Copied {n_figs} figures to Drive')

# Copy SSL v2 checkpoints
for ckpt_name in ['study0_ssl_experiment_v2', 'study0_ssl_experiment']:
    ckpt_src = Path(f'models/checkpoints/{ckpt_name}')
    if ckpt_src.exists():
        drive_ckpt = f'/content/drive/MyDrive/Fin-JEPA/models/checkpoints/{ckpt_name}'
        shutil.copytree(str(ckpt_src), drive_ckpt, dirs_exist_ok=True)
        print(f'Copied {ckpt_name} checkpoints to Drive')

print('\nAll results saved to Google Drive!')


In [ ]:
import json
from datetime import datetime

report = {
    'verified_at': datetime.now().isoformat(),
    'config_fingerprints_match': True,  # would have raised above if not
    'seed_42_consistent': True,  # would have raised above if not
    'result_files': {
        'final_benchmark': str(RESULTS_DIR / 'final_benchmark.json'),
        'multiseed_results': str(RESULTS_DIR / 'multiseed_results.json'),
        'ssl_experiment': str(RESULTS_DIR / 'ssl_experiment_results_v2.json'),
    },
}

report_path = RESULTS_DIR / 'verification_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Verification report saved to {report_path}')